<a href="https://colab.research.google.com/github/M1croZavr/progressive_cramming/blob/feature%2Fdemonstration-notebook/notebooks/progressive_cramming_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Progressive Cramming

**Compress a span of text and watch it reconstruct.** Hands-on companion to the paper *Progressive Cramming: Reliable Token Compression and What It Reveals* — runs on a free **Colab T4 GPU**.

> ⚠️ **Requires a GPU.** Runtime ▸ Change runtime type ▸ T4 GPU, then run cells top to bottom.

**Sections.** 1 · Setup &middot; 2 · What is cramming? &middot; 3 · Gallery &middot; 4 · Total vs Progressive &middot; 5 · Compress your own

---
## 1 · Setup

Install the package and the small UI dependency, then load the frozen model. The first install pulls `transformers`/`torch` and takes a couple of minutes.

In [1]:
GALLERY_ID       = "LarryLovestein/progressive_cramming_demo_gallery"  # pre-computed embeddings
REPO_URL         = "https://github.com/FusionBrainLab/progressive_cramming"
SLIDES_URL       = "https://fusionbrainlab.github.io/progressive_cramming/slides/"

In [2]:
# Project package
%pip install -q "git+https://github.com/FusionBrainLab/progressive_cramming.git" ipywidgets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00


In [3]:
import torch
from datasets import load_dataset

from progressive_cramming.demo import load_frozen_model
from progressive_cramming.demo.notebook import display_gallery
from progressive_cramming.demo.notebook import display_side_by_side
from progressive_cramming.demo.notebook import display_interactive_compress

In [4]:
assert torch.cuda.is_available(), "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then re-run!"
print("GPU:", torch.cuda.get_device_name(0))

# Carefully export HF_TOKEN! export HF_TOKEN="***""
llama_model, llama_tokenizer = load_frozen_model("unsloth/Llama-3.2-1B", dtype="float16")
print(f"Loaded unsloth/Llama-3.2-1B | hidden size: {llama_model.config.hidden_size}")

smollm_model, smollm_tokenizer = load_frozen_model("HuggingFaceTB/SmolLM2-360M", dtype="float16")
print(f"Loaded HuggingFaceTB/SmolLM2-360M | hidden size: {smollm_model.config.hidden_size}")

GPU: Tesla T4


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM2-360M | hidden size: 960


---
## 2 · What is cramming?

**Cramming** freezes a pretrained language model and optimises *one* input embedding — by gradient descent — until the frozen model reconstructs the original tokens from it. Model weights never change.

**Progressive cramming** grows the target span stage by stage and stops at the **compression horizon**: the longest prefix that reconstructs *exactly*. Each new stage **warm-starts** from the previous embedding.

<p align="center">
<img src="https://raw.githubusercontent.com/FusionBrainLab/progressive_cramming/main/page/assets/fig_progressive.png" width="720" alt="Progressive cramming schematic">
</p>

📄 [Paper & code](https://github.com/FusionBrainLab/progressive_cramming) &middot; 🎬 [Slides](https://fusionbrainlab.github.io/progressive_cramming/slides/) &middot; 🗂 [Trajectories dataset](https://huggingface.co/datasets/mrsndmn/progressive_cramming_trajectories)

---
## 3 · Pre-compressed gallery

Five paragraphs from different domains — each crammed into **one memory embedding** *(cached on the Hub so you don't wait for optimisation)*. Click **▶ Reconstruct** to greedy-decode it back. Green = matches the original at that position.

In [6]:
# Gallery examples trained on unsloth/Llama-3.2-1B
gallery_dataset = load_dataset(GALLERY_ID, split="train")
gallery_rows = [row for row in gallery_dataset if row["kind"] == "gallery"]

README.md:   0%|          | 0.00/936 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/147k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9 [00:00<?, ? examples/s]

In [7]:
display_gallery(gallery_rows, model=llama_model, tokenizer=llama_tokenizer)

---
## 4 · Total vs Progressive cramming

Two side-by-side pairs on the **same passage** (PG19 sample #7), illustrating two TC failure modes:

- **Llama-3.2-1B (128 tokens).** TC's one residual error lands on the very first content token; greedy decoding diverges immediately. PC reconstructs all 128 tokens.
- **SmolLM2-360M (32 tokens).** TC produces the first ~6 tokens correctly, then drifts into a *coherent but different* enumeration ("four things… should not have to worry about"). PC reconstructs all 32 tokens.

Both TC embeddings reach ≥ 0.99 teacher-forced accuracy. The difference is **what greedy decoding does** with that residual error: PC has none, TC has one.

In [8]:
side_by_side_rows = [row for row in gallery_dataset if row["kind"] == "tc_pc"]

In [9]:
display_side_by_side(
    side_by_side_rows,
    models={
        "unsloth/Llama-3.2-1B": (llama_model, llama_tokenizer),
        "HuggingFaceTB/SmolLM2-360M": (smollm_model, smollm_tokenizer),
    },
)

---
## 5 · Compress your own text

Type a short text, pick one of the loaded models, hit **▶ Compress**. The widget runs *progressive cramming* live: loss + teacher-forced reconstruction on the left, the embedding's PCA trajectory on the right — points coloured by current stage so the warm-start jumps between stages are visible. The run stops at the **compression horizon** and the final decode appears below with the same green / red diff as §3 and §4.

In [5]:
interactive_models = {
    "Llama-3.2-1B": (llama_model, llama_tokenizer),
    "SmolLM2-360M": (smollm_model, smollm_tokenizer),
}
MAX_SEQ_LEN = 128  # interactive cramming cap (keeps the demo snappy on a T4)
display_interactive_compress(interactive_models, max_seq_len=MAX_SEQ_LEN)